In [ ]:
%pip -q install yfinance scikit-learn

In [ ]:
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

TICKER = "AAPL"       # Match the ticker used in the RNN practical
PERIOD = "5y"
WINDOW = 30
UNITS = 50
EPOCHS = 30
BATCH_SIZE = 32

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
raw = yf.download(TICKER, period=PERIOD, interval="1d",
                  auto_adjust=False, progress=False)
if raw.empty:
    raise ValueError(f"No data returned for {TICKER}. Check ticker/network.")
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

FEATURES = ["Open", "High", "Low", "Close", "Volume"]
missing = [c for c in FEATURES if c not in raw.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")

data = raw[FEATURES].dropna().copy()
data.index = pd.to_datetime(data.index)
display(data.head())
print("Rows:", len(data), "| Date range:", data.index.min().date(), "to", data.index.max().date())

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax[0].plot(data.index, data["Close"])
ax[0].set_title(f"{TICKER} Closing Price")
ax[0].set_ylabel("Price")
ax[0].grid(alpha=.3)
ax[1].plot(data.index, data["Volume"])
ax[1].set_title("Trading Volume")
ax[1].set_ylabel("Volume")
ax[1].grid(alpha=.3)
plt.tight_layout()
plt.show()

In [ ]:
values = data[FEATURES].to_numpy(dtype="float32")
n = len(values)
train_end, val_end = int(.70*n), int(.85*n)
if train_end <= WINDOW or val_end <= train_end or n-val_end < 2:
    raise ValueError("Not enough historical rows for this split.")

scaler = MinMaxScaler()
scaler.fit(values[:train_end])
scaled = scaler.transform(values)
close_idx = FEATURES.index("Close")

print("Train rows:", train_end, "Validation rows:", val_end-train_end,
      "Test rows:", n-val_end)

In [ ]:
def make_windows(a, start, end):
    X, y, idx = [], [], []
    for t in range(max(WINDOW, start), end):
        X.append(a[t-WINDOW:t, :])
        y.append(a[t, close_idx])
        idx.append(t)
    return (np.asarray(X, np.float32),
            np.asarray(y, np.float32),
            np.asarray(idx, np.int64))

X_train, y_train, train_idx = make_windows(scaled, WINDOW, train_end)
X_val, y_val, val_idx = make_windows(scaled, train_end, val_end)
X_test, y_test, test_idx = make_windows(scaled, val_end, n)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
def build_rnn():
    m = tf.keras.Sequential([
        tf.keras.layers.Input((WINDOW, len(FEATURES))),
        tf.keras.layers.SimpleRNN(UNITS, activation="tanh"),
        tf.keras.layers.Dense(1)
    ], name="SimpleRNN_50")
    m.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss="mse", metrics=["mae"])
    return m

def build_gru():
    m = tf.keras.Sequential([
        tf.keras.layers.Input((WINDOW, len(FEATURES))),
        tf.keras.layers.GRU(UNITS, activation="tanh"),
        tf.keras.layers.Dense(1)
    ], name="GRU_50")
    m.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss="mse", metrics=["mae"])
    return m

rnn_model, gru_model = build_rnn(), build_gru()
print("SimpleRNN parameters:", rnn_model.count_params())
print("GRU parameters:", gru_model.count_params())

In [ ]:
class EpochTimer(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []
    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.perf_counter() - self.t0)

def fit_and_time(model):
    timer = EpochTimer()
    callbacks = [
        timer,
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=.5, patience=3, min_lr=1e-6)
    ]
    t0 = time.perf_counter()
    hist = model.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=False,
        callbacks=callbacks, verbose=1
    )
    return hist, time.perf_counter()-t0, timer.times

tf.keras.utils.set_random_seed(SEED)
rnn_history, rnn_seconds, rnn_epoch_seconds = fit_and_time(rnn_model)
tf.keras.utils.set_random_seed(SEED)
gru_history, gru_seconds, gru_epoch_seconds = fit_and_time(gru_model)
print(f"RNN total: {rnn_seconds:.2f}s | GRU total: {gru_seconds:.2f}s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(rnn_history.history["loss"], label="Train")
ax[0].plot(rnn_history.history["val_loss"], label="Validation")
ax[0].set_title("SimpleRNN loss"); ax[0].set_xlabel("Epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(gru_history.history["loss"], label="Train")
ax[1].plot(gru_history.history["val_loss"], label="Validation")
ax[1].set_title("GRU loss"); ax[1].set_xlabel("Epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4))
plt.plot(rnn_history.history["val_loss"], marker="o", label="RNN validation")
plt.plot(gru_history.history["val_loss"], marker="o", label="GRU validation")
plt.title("Validation loss comparison"); plt.xlabel("Epoch"); plt.ylabel("MSE")
plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
def inverse_close(a):
    tmp = np.zeros((len(a), len(FEATURES)), dtype=np.float32)
    tmp[:, close_idx] = np.asarray(a).reshape(-1)
    return scaler.inverse_transform(tmp)[:, close_idx]

rnn_scaled = rnn_model.predict(X_test, verbose=0).reshape(-1)
gru_scaled = gru_model.predict(X_test, verbose=0).reshape(-1)
actual = inverse_close(y_test)
rnn_pred, gru_pred = inverse_close(rnn_scaled), inverse_close(gru_scaled)

def metrics(y, p):
    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": np.sqrt(mean_squared_error(y, p)),
        "R2": r2_score(y, p)
    }

performance = pd.DataFrame(
    [metrics(actual, rnn_pred), metrics(actual, gru_pred)],
    index=["SimpleRNN", "GRU"]
)
display(performance)

In [ ]:
dates = data.index[test_idx]
plt.figure(figsize=(14, 6))
plt.plot(dates, actual, label="Actual Close", linewidth=2)
plt.plot(dates, rnn_pred, label="SimpleRNN prediction")
plt.plot(dates, gru_pred, label="GRU prediction")
plt.title(f"{TICKER}: Actual vs RNN and GRU predictions")
plt.xlabel("Date"); plt.ylabel("Closing price")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
def summary_row(name, model, hist, seconds, epoch_times):
    v = hist.history["val_loss"]
    return {
        "Model": name,
        "Parameters": model.count_params(),
        "Epochs completed": len(hist.history["loss"]),
        "Best validation epoch": int(np.argmin(v)) + 1,
        "Best validation loss": float(np.min(v)),
        "Total training seconds": seconds,
        "Average epoch seconds": float(np.mean(epoch_times))
    }

efficiency = pd.DataFrame([
    summary_row("SimpleRNN", rnn_model, rnn_history, rnn_seconds, rnn_epoch_seconds),
    summary_row("GRU", gru_model, gru_history, gru_seconds, gru_epoch_seconds)
]).set_index("Model")
display(efficiency)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(rnn_epoch_seconds)+1), rnn_epoch_seconds, marker="o", label="RNN")
plt.plot(range(1, len(gru_epoch_seconds)+1), gru_epoch_seconds, marker="o", label="GRU")
plt.title("Training time per epoch"); plt.xlabel("Epoch"); plt.ylabel("Seconds")
plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
last_window = scaled[-WINDOW:, :].reshape(1, WINDOW, len(FEATURES))
rnn_next = inverse_close(rnn_model.predict(last_window, verbose=0).reshape(-1))[0]
gru_next = inverse_close(gru_model.predict(last_window, verbose=0).reshape(-1))[0]
print("Ticker:", TICKER, "| Latest data date:", data.index[-1].date())
print(f"SimpleRNN estimate: {rnn_next:.4f}")
print(f"GRU estimate:       {gru_next:.4f}")
print("These are educational model estimates, not financial advice.")

In [ ]:
print("RNN metrics:", metrics(actual, rnn_pred))
print("GRU metrics:", metrics(actual, gru_pred))
print("Lower test RMSE:",
      "GRU" if performance.loc["GRU", "RMSE"] < performance.loc["SimpleRNN", "RMSE"]
      else "SimpleRNN")
print("Shorter total training time:",
      "GRU" if gru_seconds < rnn_seconds else "SimpleRNN")
print("Use validation curves, test metrics, and epoch timing together when drawing conclusions.")